# Personalized TED Talks Recommendation System Using Machine Learning

## 1. Project Overview

The objective of this project is to develop a personalized TED Talks recommendation system using content-based filtering.

The system will recommend talks that are similar to a talk selected by the user. It will use information such as titles, descriptions, tags, and transcripts to represent the content of each talk.

In the initial stage, the available datasets will be explored to understand their structure, features, and suitability for the recommendation system.

## 2. Dataset Description

The project uses two datasets:

1. **TED Talks Dataset:** Contains talk titles, descriptions, speakers, tags, event details, view counts, and other metadata.
2. **TED Transcripts Dataset:** Contains transcripts of TED Talks along with their corresponding URLs.

The `url` column will be examined as the potential common identifier for connecting the two datasets.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
df1 = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\raw\ted_talks.csv")

In [26]:
df1.head(2)

,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views
0,4553,Sir Ken Robinson makes an entertaining and pro...,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",Author/educator,"['children', 'creativity', 'culture', 'dance',...",Do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_sc...,47227110
1,265,With the same humor and humanity he exuded in ...,977,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",Climate advocate,"['alternative energy', 'cars', 'climate change...",Averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_...,3200520


In [27]:
df2 = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\raw\ted_transcripts.csv")

In [28]:
df2.head(2)

,transcript,url
0,Good morning. How are you?(Laughter)It's been ...,https://www.ted.com/talks/ken_robinson_says_sc...
1,"Thank you so much, Chris. And it's truly a gre...",https://www.ted.com/talks/al_gore_on_averting_...


In [29]:
print("Main(df1) dataset shape:", df1.shape)
print("Transcript(df2) dataset shape:", df2.shape)

Main(df1) dataset shape: (2550, 17)
Transcript(df2) dataset shape: (2467, 2)


In [30]:
print("df1 dataset columns:")
print(df1.columns.tolist())

print("\ndf2 dataset columns:")
print(df2.columns.tolist())

df1 dataset columns:
['comments', 'description', 'duration', 'event', 'film_date', 'languages', 'main_speaker', 'name', 'num_speaker', 'published_date', 'ratings', 'related_talks', 'speaker_occupation', 'tags', 'title', 'url', 'views']

df2 dataset columns:
['transcript', 'url']


In [31]:
print("df1 information:")
df1.info()

print("="*60)
print("\ndf2 information:")
df2.info()

df1 information:
<class 'pandas.DataFrame'>
RangeIndex: 2550 entries, 0 to 2549
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   comments            2550 non-null   int64
 1   description         2550 non-null   str  
 2   duration            2550 non-null   int64
 3   event               2550 non-null   str  
 4   film_date           2550 non-null   int64
 5   languages           2550 non-null   int64
 6   main_speaker        2550 non-null   str  
 7   name                2550 non-null   str  
 8   num_speaker         2550 non-null   int64
 9   published_date      2550 non-null   int64
 10  ratings             2550 non-null   str  
 11  related_talks       2550 non-null   str  
 12  speaker_occupation  2544 non-null   str  
 13  tags                2550 non-null   str  
 14  title               2550 non-null   str  
 15  url                 2550 non-null   str  
 16  views               2550 non-null   

In [32]:
print("Count of null values in every column in df1")
df1.isnull().sum()

Count of null values in every column in df1


comments              0
description           0
duration              0
event                 0
film_date             0
languages             0
main_speaker          0
name                  0
num_speaker           0
published_date        0
ratings               0
related_talks         0
speaker_occupation    6
tags                  0
title                 0
url                   0
views                 0
dtype: int64

In [33]:
print("Count of null values in every column in df2")
df1.isnull().sum()

Count of null values in every column in df2


comments              0
description           0
duration              0
event                 0
film_date             0
languages             0
main_speaker          0
name                  0
num_speaker           0
published_date        0
ratings               0
related_talks         0
speaker_occupation    6
tags                  0
title                 0
url                   0
views                 0
dtype: int64

In [34]:
print("Count of null values in every column in df2")
df2.isnull().sum()

Count of null values in every column in df2


transcript    0
url           0
dtype: int64

In [35]:
print("Number of Duplicated rows in df1:",df1.duplicated().sum())

Number of Duplicated rows in df1: 0


In [36]:
print("Number of Duplicated rows in df2:",df2.duplicated().sum())

Number of Duplicated rows in df2: 3


In [37]:
print("Unique talk IDs in df1:", df1["url"].nunique())
print("Duplicate talk IDs in df1:", df1["url"].duplicated().sum())

Unique talk IDs in df1: 2550
Duplicate talk IDs in df1: 0


In [38]:
print("Unique talk IDs in df2:", df2["url"].nunique())
print("Duplicate talk IDs in df2:", df2["url"].duplicated().sum())

Unique talk IDs in df2: 2464
Duplicate talk IDs in df2: 3


In [39]:
duplicate_transcripts = df2[df2["url"].duplicated(keep=False)].sort_values("url")

display(duplicate_transcripts[["url", "transcript"]])

,url,transcript
1109,https://www.ted.com/talks/brene_brown_listenin...,I'm going to tell you a little bit about my TE...
1116,https://www.ted.com/talks/brene_brown_listenin...,I'm going to tell you a little bit about my TE...
1107,https://www.ted.com/talks/jonathan_haidt_human...,I have a question for you: Are you religious? ...
1114,https://www.ted.com/talks/jonathan_haidt_human...,I have a question for you: Are you religious? ...
1108,https://www.ted.com/talks/rob_reid_the_8_billi...,The recent debate over copyright laws like SOP...
1115,https://www.ted.com/talks/rob_reid_the_8_billi...,The recent debate over copyright laws like SOP...


In [40]:
df2.iloc[4,0]

'About 10 years ago, I took on the task to teach global development to Swedish undergraduate students. That was after having spent about 20 years together with African institutions studying hunger in Africa, so I was sort of expected to know a little about the world. And I started in our medical university, Karolinska Institute, an undergraduate course called Global Health. But when you get that opportunity, you get a little nervous. I thought, these students coming to us actually have the highest grade you can get in Swedish college systems — so I thought, maybe they know everything I\'m going to teach them about. So I did a pre-test when they came. And one of the questions from which I learned a lot was this one: "Which country has the highest child mortality of these five pairs?"I put them together, so that in each pair of country, one has twice the child mortality of the other. And this means that it\'s much bigger a difference than the uncertainty of the data. I won\'t put you at 

In [41]:
df2.iloc[1,0]

'Thank you so much, Chris. And it\'s truly a great honor to have the opportunity to come to this stage twice; I\'m extremely grateful. I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night. And I say that sincerely, partly because (Mock sob) I need that.(Laughter)Put yourselves in my position.(Laughter)I flew on Air Force Two for eight years.(Laughter)Now I have to take off my shoes or boots to get on an airplane!(Laughter)(Applause)I\'ll tell you one quick story to illustrate what that\'s been like for me.(Laughter)It\'s a true story — every bit of this is true.Soon after Tipper and I left the — (Mock sob) White House —(Laughter)we were driving from our home in Nashville to a little farm we have 50 miles east of Nashville. Driving ourselves.(Laughter)I know it sounds like a little thing to you, but —(Laughter)I looked in the rear-view mirror and all of a sudden it just hit me. There was no motorcade

In [42]:
df2.iloc[1,1]

'https://www.ted.com/talks/al_gore_on_averting_climate_crisis\n'

In [43]:
df1["event"].nunique()

355

In [44]:
(df1["views"] < 0).sum()

np.int64(0)

In [46]:
df1['description'][0]

'Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (rather than undermines) creativity.'

In [47]:
df1['title'][0]

'Do schools kill creativity?'

In [48]:
df1['tags'][0]

"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']"

In [49]:
df1['ratings'][0]

"[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingenious', 'count': 6073}, {'id': 3, 'name': 'Courageous', 'count': 3253}, {'id': 11, 'name': 'Longwinded', 'count': 387}, {'id': 2, 'name': 'Confusing', 'count': 242}, {'id': 8, 'name': 'Informative', 'count': 7346}, {'id': 22, 'name': 'Fascinating', 'count': 10581}, {'id': 21, 'name': 'Unconvincing', 'count': 300}, {'id': 24, 'name': 'Persuasive', 'count': 10704}, {'id': 23, 'name': 'Jaw-dropping', 'count': 4439}, {'id': 25, 'name': 'OK', 'count': 1174}, {'id': 26, 'name': 'Obnoxious', 'count': 209}, {'id': 10, 'name': 'Inspiring', 'count': 24924}]"

In [50]:
df1['related_talks'][0]

'[{\'id\': 865, \'hero\': \'https://pe.tedcdn.com/images/ted/172559_800x600.jpg\', \'speaker\': \'Ken Robinson\', \'title\': \'Bring on the learning revolution!\', \'duration\': 1008, \'slug\': \'sir_ken_robinson_bring_on_the_revolution\', \'viewed_count\': 7266103}, {\'id\': 1738, \'hero\': \'https://pe.tedcdn.com/images/ted/de98b161ad1434910ff4b56c89de71af04b8b873_1600x1200.jpg\', \'speaker\': \'Ken Robinson\', \'title\': "How to escape education\'s death valley", \'duration\': 1151, \'slug\': \'ken_robinson_how_to_escape_education_s_death_valley\', \'viewed_count\': 6657572}, {\'id\': 2276, \'hero\': \'https://pe.tedcdn.com/images/ted/3821f3728e0b755c7b9aea2e69cc093eca41abe1_2880x1620.jpg\', \'speaker\': \'Linda Cliatt-Wayman\', \'title\': \'How to fix a broken school? Lead fearlessly, love hard\', \'duration\': 1027, \'slug\': \'linda_cliatt_wayman_how_to_fix_a_broken_school_lead_fearlessly_love_hard\', \'viewed_count\': 1617101}, {\'id\': 892, \'hero\': \'https://pe.tedcdn.com/ima

In [51]:
df1['languages'].nunique()

66

In [52]:
df1['languages'].unique()

array([60, 43, 26, 35, 48, 36, 31, 19, 32, 27, 20, 24, 25, 22, 33, 23, 29,
       37, 45, 21, 28, 41, 66, 34,  0, 18, 17, 30, 16, 38, 42, 44, 39, 40,
       47, 49, 61, 52, 51, 53, 56, 64, 46, 69, 65, 58, 57, 55, 54, 59, 72,
       13,  3, 11, 10, 14, 15,  5,  7,  8,  9, 12,  6,  4,  2,  1])

In [54]:
df1['num_speaker'].nunique()

5

In [55]:
df1['num_speaker'].unique()

array([1, 2, 3, 4, 5])

# Dataset Inspection Report

## 1. Dataset Name and Source

The dataset used for this project is the **TED Talks dataset by rounakbanik**, obtained from Kaggle.

Dataset link: https://www.kaggle.com/datasets/rounakbanik/ted-talks

The project uses two CSV files:

- `ted_talks.csv` — contains TED Talk information and metadata.
- `ted_transcripts.csv` — contains TED Talk transcripts and their corresponding URLs.

## 2. Dataset Dimensions

The original datasets were loaded and inspected without modifying their contents.

- **Main dataset:** 2,550 rows and 17 columns.
- **Transcript dataset:** 2,467 rows and 2 columns.

The main dataset contains talk details, descriptions, speaker information, tags, and numerical metadata. The transcript dataset contains the spoken content of talks and their corresponding URLs.

## 3. Dataset Structure and Features

The main dataset includes columns such as `title`, `description`, `tags`, `main_speaker`, `speaker_occupation`, `event`, `views`, `duration`, and `url`.

The transcript dataset contains two columns:

- `transcript`: The spoken content of a TED Talk.
- `url`: The URL associated with the talk.

The initial inspection examined the dimensions, column names, data types, and sample records to understand the structure of both datasets.

## 4. Missing Values

The initial inspection identified **6 missing values** in the `speaker_occupation` column of the main dataset. All other main-dataset columns had zero missing values.

The transcript dataset had zero missing values in both the `transcript` and `url` columns.

The missing speaker occupations will be handled during the data-cleaning stage by replacing them with `"Unknown"`, provided the values remain missing in the original data.

## 5. Duplicate Records and Identifiers

The main dataset contains:

- Exact duplicate rows: 0
- Duplicate URLs: 0

The original transcript dataset contains:

- Exact duplicate rows: 3
- Duplicate URLs: 3

The transcript duplicates were identified during the initial inspection. They will be handled during the data-cleaning stage.

Since the datasets do not provide an explicit `talk_id` column, the `url` column is used as the candidate identifier for matching talks across the two files.

## 6. Transcript Availability

The URL comparison between the datasets produced the following results:

- Matching URLs: 2,464
- Main-dataset URLs without matching transcripts: 86
- Transcript URLs absent from the main dataset: 0

These results indicate that transcripts are available for most, but not all, talks in the main dataset.

The 86 talks without matching transcripts can still be considered for recommendations using their available titles, descriptions, and tags.

The transcript dataset contains duplicate rows and repeated URLs, so the transcript records will be checked during data cleaning before the final merge.

## 7. Features Relevant to the Recommendation System

The initial inspection identified the following fields as useful for the content-based recommendation system:

- `title`: The title of the talk.
- `description`: A short description of the talk.
- `tags`: Topics and keywords associated with the talk.
- `transcript`: The spoken content of the talk.
- `main_speaker`: The speaker's name.
- `speaker_occupation`: The speaker's occupation, where available.

The primary text fields will be `title`, `description`, `tags`, and `transcript`. Speaker information can be retained as metadata.

The planned recommendation approach uses TF-IDF to convert textual content into numerical vectors and cosine similarity to identify talks with similar content.

## 8. Dataset Limitations

The initial inspection identified the following limitations:

1. The datasets do not contain an explicit `talk_id` column; URLs are used as candidate identifiers.
2. Some talks in the main dataset do not have matching transcripts.
3. The original transcript dataset contains three exact duplicate rows and three repeated URL entries.
4. The main dataset contains six missing speaker occupation values.
5. The datasets do not provide user-interaction records required for a conventional collaborative-filtering approach.

These limitations will be considered during data cleaning and recommendation-system development.

## 9. Suitability for a Content-Based Recommendation Prototype

The datasets provide relevant information for developing an initial content-based TED Talks recommendation system.

The main dataset supplies talk titles, descriptions, tags, speaker information, and other metadata. The transcript dataset supplies additional textual information for talks with available transcripts.

The datasets can be matched using the `url` column after duplicate records have been investigated and handled. A left join will be used during data cleaning to preserve all 2,550 main-dataset talks, including those without transcripts.

The combined text will subsequently be prepared for TF-IDF vectorization, and cosine similarity will be used to generate recommendations.

Overall, the initial inspection indicates that the datasets are suitable for beginning development of the proposed content-based recommendation prototype.

## 10. Conclusion

The initial dataset inspection established the structure, dimensions, key features, missing values, duplicate records, and transcript coverage of the two TED Talks datasets.

The main dataset contains 2,550 talks, of which 2,464 have matching transcript URLs. The identified missing occupation values and transcript duplicates will be addressed during data cleaning.

The next stage is detailed exploratory data analysis, documented in `02_eda_data_quality.ipynb`, followed by data cleaning, text preprocessing, dataset merging, TF-IDF vectorization, cosine similarity, and recommendation-system development.